# Day 09. Exercise 01
# Gridsearch

## 0. Imports

In [24]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid, cross_val_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from tqdm.notebook import tqdm
from sklearn.metrics import accuracy_score

## 1. Preprocessing

1. Read the file [`day-of-week-not-scaled.csv`](https://drive.google.com/file/d/1AlGvsJDSzPT_70caausx8bFuupIEZkfh/view?usp=sharing). It is similar to the one from the previous exercise, but this time we did not scale continuous features (we are not going to use logreg anymore). Don't forget to enrich the table with the 'dayofweek' column from the previous day's .csv-file.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [25]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
df

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [26]:
dayofweek = pd.read_csv('../data/dayofweek.csv')
df['dayofweek'] = dayofweek.dayofweek
del dayofweek

In [27]:
X = df.drop(columns=['dayofweek'])
y = df.dayofweek

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)

## 2. SVM gridsearch

1. Using `GridSearchCV` try different parameters of kernel (`linear`, `rbf`, `sigmoid`), C (`0.01`, `0.1`, `1`, `1.5`, `5`, `10`), gamma (`scale`, `auto`), class_weight (`balanced`, `None`) use `random_state=21` and `probability=True` and get the best combination of them in terms of accuracy.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`. Check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [29]:
svc_param_grid = {
    'kernel':['linear', 'rbf', 'sigmoid'],
    'C' : [0.01, 0.1, 1, 1.5, 5, 10],
    'gamma' : ['scale', 'auto'],
    'class_weight' : ['balanced', None]
}
svc_model = SVC(random_state=21, probability=True)
svc_grid_search = GridSearchCV(svc_model, svc_param_grid, n_jobs=-1, scoring='accuracy')

In [30]:
svc_grid_search.fit(X_train, y_train)

GridSearchCV(estimator=SVC(probability=True, random_state=21), n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 1.5, 5, 10],
                         'class_weight': ['balanced', None],
                         'gamma': ['scale', 'auto'],
                         'kernel': ['linear', 'rbf', 'sigmoid']},
             scoring='accuracy')

In [31]:
print(f'Лучшие параметры: {svc_grid_search.best_params_}')
print(f'Лучший score: {svc_grid_search.best_score_}')

Лучшие параметры: {'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel': 'rbf'}
Лучший score: 0.8761090458488228


In [32]:
svc_results = pd.DataFrame(svc_grid_search.cv_results_)
svc_results = svc_results.sort_values('rank_test_score')
svc_results.head(7)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_C,param_class_weight,param_gamma,param_kernel,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
70,0.872186,0.031986,0.050465,0.008463,10,None,auto,rbf,"{'C': 10, 'class_weight': None, 'gamma': 'auto...",0.900000,0.848148,0.885185,0.884758,0.862454,0.876109,0.018419,1
64,0.849576,0.015440,0.043804,0.003183,10,balanced,auto,rbf,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.877778,0.851852,0.862963,0.873606,0.851301,0.863500,0.010870,2
58,0.727372,0.013357,0.049913,0.010092,5,None,auto,rbf,"{'C': 5, 'class_weight': None, 'gamma': 'auto'...",0.825926,0.811111,0.818519,0.821561,0.802974,0.816018,0.008116,3
52,0.744478,0.028911,0.045237,0.002274,5,balanced,auto,rbf,"{'C': 5, 'class_weight': 'balanced', 'gamma': ...",0.844444,0.785185,0.792593,0.817844,0.802974,0.808608,0.021007,4
63,65.773248,4.322989,0.014610,0.004690,10,balanced,auto,linear,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.729630,0.700000,0.755556,0.754647,0.665428,0.721052,0.034438,5
60,66.893821,5.648079,0.010605,0.002171,10,balanced,scale,linear,"{'C': 10, 'class_weight': 'balanced', 'gamma':...",0.729630,0.700000,0.755556,0.754647,0.665428,0.721052,0.034438,5
66,53.456656,4.518669,0.009300,0.003126,10,None,scale,linear,"{'C': 10, 'class_weight': None, 'gamma': 'scal...",0.737037,0.711111,0.707407,0.743494,0.698885,0.719587,0.017463,7


## 3. Decision tree

1. Using `GridSearchCV` try different parameters of `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use `random_state=21`.
2. Create a dataframe from the results of the gridsearch and sort it ascendingly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [33]:
tree_param_grid = {
    'max_depth' : range(1,50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini']
}
tree_model = DecisionTreeClassifier(random_state=21)
tree_grid_search = GridSearchCV(tree_model, tree_param_grid, scoring='accuracy', n_jobs=-1)

In [34]:
tree_grid_search.fit(X_train,  y_train)

GridSearchCV(estimator=DecisionTreeClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50)},
             scoring='accuracy')

In [35]:
print(f'Лучшие параметры: {tree_grid_search.best_params_}')
print(f'Лусший score: {tree_grid_search.best_score_}')

Лучшие параметры: {'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22}
Лусший score: 0.8731212997384002


In [36]:
tree_results = pd.DataFrame(tree_grid_search.cv_results_)
tree_results.sort_values(axis=0, by=['rank_test_score'], inplace=True)
tree_results.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
70,0.008703,0.001469,0.003506,0.001556,balanced,gini,22,"{'class_weight': 'balanced', 'criterion': 'gin...",0.885185,0.862963,0.903704,0.881041,0.832714,0.873121,0.023998,1
69,0.009999,0.001266,0.003801,0.001833,balanced,gini,21,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.859259,0.903704,0.884758,0.828996,0.873121,0.026300,2
97,0.012752,0.002123,0.003820,0.000753,balanced,gini,49,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3
95,0.009200,0.001166,0.003600,0.001201,balanced,gini,47,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3
94,0.008998,0.000894,0.003402,0.000489,balanced,gini,46,"{'class_weight': 'balanced', 'criterion': 'gin...",0.888889,0.866667,0.903704,0.873606,0.832714,0.873116,0.023911,3


## 4. Random forest

1. Using `GridSearchCV` try different parameters of `n_estimators` (`5`, `10`, `50`, `100`), `max_depth` (from `1` to `49`), `class_weight` (`balanced`, `None`) and `criterion` (`entropy` and `gini`) and get the best combination of them in terms of accuracy. Use random_state=21.
2. Create a dataframe from the results of the gridsearch and sort it ascendengly by the `rank_test_score`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [37]:
forest_param_grid = {
    'n_estimators' : [5,10,50,100],
    'max_depth' : range(1,50),
    'class_weight' : ['balanced', None],
    'criterion' : ['entropy', 'gini'],
    'random_state' : [21]
}
forest_model = RandomForestClassifier(random_state=21)
forest_grid_search = GridSearchCV(forest_model, forest_param_grid, scoring='accuracy', n_jobs=-1)

In [38]:
forest_grid_search.fit(X_train, y_train)

GridSearchCV(estimator=RandomForestClassifier(random_state=21), n_jobs=-1,
             param_grid={'class_weight': ['balanced', None],
                         'criterion': ['entropy', 'gini'],
                         'max_depth': range(1, 50),
                         'n_estimators': [5, 10, 50, 100],
                         'random_state': [21]},
             scoring='accuracy')

In [39]:
print(f'Best params: {forest_grid_search.best_params_}')
print(f'Best score: {forest_grid_search.best_score_}')

Best params: {'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators': 50, 'random_state': 21}
Best score: 0.9042902381935839


In [40]:
forest_result = pd.DataFrame(forest_grid_search.cv_results_)
forest_result.sort_values('rank_test_score', inplace=True)
forest_result.head()

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_class_weight,param_criterion,param_max_depth,param_n_estimators,param_random_state,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
698,0.158490,0.009311,0.011819,0.003057,None,gini,28,50,21,"{'class_weight': None, 'criterion': 'gini', 'm...",0.922222,0.900000,0.907407,0.903346,0.888476,0.904290,0.010961,1
711,0.348278,0.016229,0.025532,0.014021,None,gini,31,100,21,"{'class_weight': None, 'criterion': 'gini', 'm...",0.918519,0.911111,0.900000,0.910781,0.877323,0.903547,0.014380,2
314,0.182466,0.003278,0.013310,0.005700,balanced,gini,30,50,21,"{'class_weight': 'balanced', 'criterion': 'gin...",0.922222,0.907407,0.881481,0.907063,0.895911,0.902817,0.013554,3
330,0.195541,0.024695,0.011608,0.002867,balanced,gini,34,50,21,"{'class_weight': 'balanced', 'criterion': 'gin...",0.922222,0.907407,0.892593,0.907063,0.884758,0.902809,0.013010,4
702,0.165372,0.007339,0.010126,0.000663,None,gini,29,50,21,"{'class_weight': None, 'criterion': 'gini', 'm...",0.918519,0.896296,0.911111,0.903346,0.884758,0.902806,0.011698,5


## 5. Progress bar

Gridsearch can be a quite long process and you may find yourself wondering when it will end.
1. Create a manual gridsearch for the same parameters values of random forest iterating through the list of the possible values and calculating `cross_val_score` for each combination. Try to increase `n_jobs`. The value `cv` for `cross_val_score` is 5.
2. Track the progress using the library `tqdm.notebook`.
3. Create a dataframe from the results of the gridsearch with the columns corresponding to the names of the parameters and `mean_accuracy` and `std_accuracy`.
4. Sort it descendingly by the `mean_accuracy`, check if there is a huge difference between different combinations (sometimes a simpler model may give a comparable result).

In [41]:
grid = ParameterGrid(forest_param_grid)
len(grid)

784

In [42]:
data = []

for params in tqdm(grid):
    d = {}
    model = RandomForestClassifier(**params)
    sc = cross_val_score(model, X_train, y_train, cv=5, n_jobs=-1)
    d = {**params, 'mean_accuracy' : np.mean(sc), 'std_accuracy' : np.std(sc)}
    data.append(d)

  0%|          | 0/784 [00:00<?, ?it/s]

In [43]:
result = pd.DataFrame(data)
result.sort_values('mean_accuracy', inplace=True, ascending=False)
result

,class_weight,criterion,max_depth,n_estimators,random_state,mean_accuracy,std_accuracy
698,None,gini,28,50,21,0.904290,0.010961
711,None,gini,31,100,21,0.903547,0.014380
314,balanced,gini,30,50,21,0.902817,0.013554
330,balanced,gini,34,50,21,0.902809,0.013010
751,None,gini,41,100,21,0.902806,0.010460
...,...,...,...,...,...,...,...
392,None,entropy,1,5,21,0.353832,0.016467
4,balanced,entropy,2,5,21,0.353110,0.021165
200,balanced,gini,2,5,21,0.346419,0.029749
196,balanced,gini,1,5,21,0.283390,0.011062


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.

In [44]:
best_model = RandomForestClassifier(
    random_state=21, 
    criterion='gini',
    max_depth=28,
    n_estimators=50
)
best_model.fit(X_train, y_train)

RandomForestClassifier(max_depth=28, n_estimators=50, random_state=21)

In [45]:
predict = best_model.predict(X_test)

In [46]:
accuracy_score(y_test, predict)

0.9289940828402367